In [ ]:

 # The key contrast is critical vs baseline_ordinary within each checkpoint: does the redundant mention elevate belief in the wonky property?

import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/pragmatics_project/results/merged_clean.csv")

id_df = df[df["step"] == "inference_derivation"]

summary = (
    id_df.groupby(["stage", "condition"])["outcome_likely"]
    .mean()
    .unstack("condition")
    .reindex(columns=["critical", "resolved", "baseline_ordinary", "baseline_wonky"])
)
summary = summary.reindex(["pretrained", "sft", "dpo"])  # training-stage order

#proportion answering 'likely' (belief in the wonky property), by checkpoint x condition
print(summary.round(3))
print()
#critical - baseline_ordinary
print((summary["critical"] - summary["baseline_ordinary"]).round(3))


#  check script_knowledge stayed near ceiling
sk_df = df[df["step"] == "script_knowledge"]
sk_summary = (
    sk_df.groupby(["stage", "condition"])["outcome_likely"].mean().unstack("condition")
)
print("\nscript_knowledge sanity check (should track ordinary=likely, wonky=unlikely reliably):")
print(sk_summary.round(3))

condition   critical  resolved  baseline_ordinary  baseline_wonky
stage                                                            
pretrained     0.458     0.792              0.417           0.833
sft            0.250     0.667              0.167           0.833
dpo            0.167     0.625              0.083           0.917

stage
pretrained    0.042
sft           0.083
dpo           0.083
dtype: float64

script_knowledge sanity check (should track ordinary=likely, wonky=unlikely reliably):
condition   baseline_ordinary  baseline_wonky  critical  resolved
stage                                                            
dpo                     0.958           0.167     0.958     0.208
pretrained              0.917           0.583     0.917     0.500
sft                     0.958           0.292     0.958     0.208


In [ ]:
#mixed analysis logistic regression

import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/pragmatics_project/results/merged_clean.csv")
sub = df[
    (df["step"] == "inference_derivation")
    & (df["condition"].isin(["critical", "baseline_ordinary"]))
].copy()

# stage as ordered categorical, pretrained as reference level
sub["stage"] = pd.Categorical(sub["stage"], categories=["pretrained", "sft", "dpo"], ordered=False)
sub["condition"] = pd.Categorical(sub["condition"], categories=["baseline_ordinary", "critical"], ordered=False)

print(f"N observations: {len(sub)} (expect 24 items x 2 conditions x 3 checkpoints = 144)")

vc_formulas = {"item": "0 + C(item_id)"}
model = BinomialBayesMixedGLM.from_formula(
    "outcome_likely ~ stage * condition",
    vc_formulas,
    data=sub,
)
result = model.fit_vb()  # variational Bayes fit
print(result.summary())

N observations in model: 144 (expect 24 items x 2 conditions x 3 checkpoints = 144)
                           Binomial Mixed GLM Results
                                   Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
---------------------------------------------------------------------------------
Intercept                             M    -0.5405   0.2153                      
stage[T.sft]                          M    -1.2772   0.3777                      
stage[T.dpo]                          M    -2.0026   0.4459                      
condition[T.critical]                 M     0.3330   0.2936                      
stage[T.sft]:condition[T.critical]    M     0.0805   0.5057                      
stage[T.dpo]:condition[T.critical]    M     0.1435   0.5783                      
item                                  V    -0.0231   0.1429 0.977   0.734   1.300
Parameter types are mean structure (M) and variance structure (V)
Variance parameters are modeled as log standard deviations

In [ ]:
# making the plots
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 16, "axes.spines.top": False, "axes.spines.right": False})

def wilson(p, n=24, z=1.96):
    center = (p + z**2/(2*n)) / (1 + z**2/n)
    half = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / (1 + z**2/n)
    return center - half, center + half

stages = ["Pretrained", "SFT", "DPO"]
baseline = [0.417, 0.167, 0.083]
critical = [0.458, 0.250, 0.167]
C_BASE, C_CRIT = "#8fa8c8", "#d9822b"

#  main result
fig, ax = plt.subplots(figsize=(8, 5.5))
x = np.arange(3); w = 0.36
for vals, off, col, lab in [(baseline, -w/2, C_BASE, "Baseline (unrelated detail)"),
                            (critical,  w/2, C_CRIT, "Critical (redundant detail)")]:
    lo = [v - wilson(v)[0] for v in vals]; hi = [wilson(v)[1] - v for v in vals]
    ax.bar(x + off, vals, w, color=col, label=lab, yerr=[lo, hi],
           capsize=4, error_kw={"lw": 1.3, "ecolor": "#444"})
    for xi, v in zip(x + off, vals):
        ax.text(xi, wilson(v)[1] + 0.015, f"{v:.2f}", ha="center", va="bottom", fontsize=13, color="black")
for i, g in enumerate([0.042, 0.083, 0.083]):
    ax.text(i, 0.72, f"gap = +{g:.3f}", ha="center", fontsize=14, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(stages)
ax.set_ylim(0, 1.0)
ax.set_ylabel("Proportion answering 'likely'\n(belief in the atypical trait)")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=1, fontsize=13)
ax.text(0.5, -0.17, "Error bars: 95% Wilson intervals, n = 24 items per bar", transform=ax.transAxes,
        ha="center", fontsize=11, color="#555")
fig.tight_layout(); fig.savefig("/content/drive/MyDrive/Colab Notebooks/pragmatics_project/results/fig1_main_result.png", dpi=300); plt.close(fig)

# validity check
ord_acc = [0.917, 0.958, 0.958]
won_acc = [0.458, 0.750, 0.813]
fig, ax = plt.subplots(figsize=(8, 5.5))
for vals, off, col, lab in [(ord_acc, -w/2, "#6aa584", "Ordinary-prior items"),
                            (won_acc,  w/2, "#b5546b", "Atypical-prior items")]:
    lo = [v - wilson(v, 48)[0] for v in vals]; hi = [wilson(v, 48)[1] - v for v in vals]
    ax.bar(x + off, vals, w, color=col, label=lab, yerr=[lo, hi],
           capsize=4, error_kw={"lw": 1.3, "ecolor": "#444"})
    for xi, v in zip(x + off, vals):
        ax.text(xi, 0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=13, color="white", fontweight="bold")
ax.axhline(0.5, ls="--", color="#777", lw=1)
ax.text(-0.55, 0.515, "chance", fontsize=11, color="#777", ha="left")
ax.set_xticks(x); ax.set_xticklabels(stages); ax.set_ylim(0, 1.15)
ax.set_ylabel("Accuracy on the prior-only\nscript-knowledge check")
ax.legend(frameon=False, loc="upper left", ncol=2, fontsize=12)
fig.tight_layout(); fig.savefig("/content/drive/MyDrive/Colab Notebooks/pragmatics_project/results/fig2_validity_check.png", dpi=300); plt.close(fig)
